# Continuity discriminator experiment (team contribution)

Adapted from continuity branch `8d29ffa` to the current production package.
Historical MPS/54-channel records are preserved in `docs/experiments/continuity/`.
New reruns use current body-only preprocessing and must not be presented as the same historical result.
Both long experiments are disabled until explicitly enabled below. No dataset is bundled.


In [ ]:

import sys
from pathlib import Path

# repo root = first folder at or above this notebook that contains src/aged
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "aged").is_dir())
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO))

import torch
from aged.critics import Critic, continuity_input, critic_loss, generator_loss

from aged.runtime import pick_device
DEVICE = str(pick_device("auto"))   # Apple GPU if present
print(torch.__version__, DEVICE)


## 1. Shape check

In [ ]:
B, T, H, D = 4, 50, 10, 54            # D is a test value; real code reads norm.n_dims
critic = Critic(D, hidden=32, architecture="direct").to(DEVICE)
obs = torch.randn(B, T, D, device=DEVICE)
fut = torch.randn(B, H, D, device=DEVICE)

p = critic(continuity_input(obs, fut))
print(p.shape, p.min().item(), p.max().item())   # torch.Size([4]), values in (0, 1)


## 2. Whole-pipeline tests

Run the unified system suite, which includes critic and gradient contracts.


In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, check=True)


In [ ]:
# Whole-pipeline tests are run in the previous cell.


## 3. Sanity-check helpers on synthetic motion

The contributor used 2,000 synthetic steps as a diagnostic. Convergence timing is data/device dependent. Set RUN_SYNTHETIC=True to run the four scenarios.


In [ ]:
OBS, FUT = 50, 10


def synthetic_motion(n, D=54, T=OBS + FUT):
    """Smooth stand-in for motion: one sine wave per channel with random
    amplitude, speed, phase and offset. Lets you test before H3.6M is downloaded."""
    t = torch.arange(T).float()[None, :, None]           # (1, T, 1)
    amp   = 0.2 + torch.rand(n, 1, D)
    speed = 0.05 + 0.3 * torch.rand(n, 1, D)              # radians per frame
    phase = 6.283 * torch.rand(n, 1, D)
    level = torch.randn(n, 1, D)
    return amp * torch.sin(speed * t + phase) + level     # (n, T, D)


def make_fake(window, kind):
    """Turn real 60-frame windows into 'generated' ones with a known defect."""
    obs, fut = window[:, :OBS], window[:, OBS:]
    if kind == "zero_velocity":           # frozen future: last frame repeated
        fut = obs[:, -1:].expand_as(fut)
    elif kind == "swapped_future":        # a REAL future, but from another window
        fut = fut.roll(1, dims=0)
    return torch.cat([obs, fut], dim=1)


VIEWS = {
    "continuity (60 frames)": lambda w: w,                # what D_c sees
    "future only (10 frames)": lambda w: w[:, OBS:],      # what D_f sees
}


def train_critic(view, kind, sample=synthetic_motion, steps=2000, batch=64,
                 D=54, hidden=128, lr=1e-3, device=DEVICE, log_every=0):
    torch.manual_seed(0)
    critic = Critic(D, hidden, architecture="direct").to(device)
    opt = torch.optim.Adam(critic.parameters(), lr=lr)
    for step in range(1, steps + 1):
        real = sample(batch).to(device)
        fake = make_fake(sample(batch), kind).to(device)
        loss, acc = critic_loss(critic, view(real), view(fake))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if log_every and step % log_every == 0:
            print(f"  step {step:5d}  loss {loss.item():.3f}  batch acc {acc:.2f}")
    with torch.no_grad():                                 # score on fresh samples
        real = sample(512).to(device)
        fake = make_fake(sample(512), kind).to(device)
        _, acc = critic_loss(critic, view(real), view(fake))
    return acc


RUN_SYNTHETIC = False
if RUN_SYNTHETIC:
    for kind in ("zero_velocity", "swapped_future"):
        for name, view in VIEWS.items():
            print(f"{kind:14s} {name:24s} accuracy {train_critic(view, kind):.2f}")


## 4. Real Human3.6M windows

Set RUN_REAL=True only after preparing the full export. Uses all six training subjects; S5 is excluded. K is inferred from current body-only preprocessing and is not fixed to the historical 54.


In [ ]:
RUN_REAL = False
if RUN_REAL:
    from aged import data as Dmod
    
    ROOT = str(REPO / "data" / "h36m")
    norm = Dmod.fit_normalisation(ROOT)                  # training subjects only (S5 excluded)
    ds = Dmod.WindowDataset(ROOT, norm, subjects=Dmod.TRAIN_SUBJECTS, length=10**6)    # same subjects, normalised
    print(norm.n_dims, "channels,", len(ds.sequences), "recordings")   # K is inferred; roots are excluded in current preprocessing


In [ ]:
def h36m_windows(n):
    """n random real 60-frame windows, normalised: (n, 60, D)."""
    idx = torch.randint(len(ds), (n,))
    pairs = [ds[int(i)] for i in idx]
    return torch.stack([torch.cat([o, t]) for o, t, _ in pairs])


## 5. Optional 4,000-step experiment and artifacts

Writes fresh results/table/figures under experiments/. Record the actual device, versions and K. Fixed seeds do not imply bit-identical results across devices or preprocessing versions; do not replace the historical contributor records with a rerun claim.


In [ ]:
if RUN_REAL:
    import json
    from datetime import datetime
    
    STEPS, HIDDEN, BATCH, LR = 4000, 128, 64, 1e-3
    KINDS = {"zero_velocity": "Zero-velocity", "swapped_future": "Swapped future"}
    
    # 1. Re-run the four cells and keep the numbers
    results = {}
    for kind in KINDS:
        results[kind] = {}
        for name, view in VIEWS.items():
            acc = train_critic(view, kind, sample=h36m_windows, D=norm.n_dims,
                               steps=STEPS, hidden=HIDDEN, batch=BATCH, lr=LR)
            results[kind][name] = acc
            print(f"{kind:14s} {name:24s} accuracy {acc:.2f}")
    
    # 2. Save the raw numbers and the settings that produced them
    out = REPO / "experiments" / f"critic-sanity-{datetime.now():%Y%m%d-%H%M%S}"
    out.mkdir(parents=True, exist_ok=True)
    record = {
        "what": "critic trained alone: real H3.6M windows vs fake windows; "
                "accuracy on 512 fresh real + 512 fresh fake",
        "data": {"root": "data/h36m", "subjects": list(Dmod.TRAIN_SUBJECTS), "channels": norm.n_dims},
        "settings": {"steps": STEPS, "hidden": HIDDEN, "batch": BATCH, "lr": LR,
                     "seed": 0, "device": DEVICE, "torch": str(torch.__version__), "critic_architecture": "direct", "preprocessing": "current body-only training statistics"},
        "accuracy": results,
    }
    (out / "results.json").write_text(json.dumps(record, indent=2))
    print("saved to", out)
    
    # 3. The 2 x 2 table, as Markdown for the report
    D_C, FUT = "continuity (60 frames)", "future only (10 frames)"
    rows = ["| Fake type | D_c: history + future (60 frames) | Future only (10 frames) |",
            "| --- | --- | --- |"]
    for kind, label in KINDS.items():
        rows.append(f"| {label} | {results[kind][D_C]:.2f} | {results[kind][FUT]:.2f} |")
    table_md = "\n".join(rows)
    (out / "table.md").write_text(table_md + "\n")
    print(table_md)
    
    # 4. The chart for the slide (PNG) and the report (PDF)
    import matplotlib.pyplot as plt
    import numpy as np
    
    SERIES = {D_C: ("D_c: history + future (60 frames)", "#2a78d6"),
              FUT: ("Future only (10 frames), as D_f sees it", "#eb6834")}
    x, w = np.arange(len(KINDS)), 0.36
    
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    for i, (name, (label, color)) in enumerate(SERIES.items()):
        vals = [results[k][name] for k in KINDS]
        bars = ax.bar(x + (i - 0.5) * w, vals, w * 0.94, color=color, label=label)
        ax.bar_label(bars, labels=[f"{v:.2f}" for v in vals], padding=3,
                     fontsize=10, color="#0b0b0b")
    
    ax.axhline(0.5, color="#8a8a85", linestyle="--", linewidth=1)
    ax.text(-0.62, 0.53, "chance", fontsize=9, color="#5f5e58")
    ax.set_xticks(x, list(KINDS.values()))
    ax.set_xlim(-0.65, len(KINDS) - 0.35)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Accuracy, real vs fake")
    ax.yaxis.grid(True, color="#e6e5e0", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_title("Only D_c catches a real future attached to the wrong history",
                 loc="left", fontsize=11)
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=1)
    fig.tight_layout()
    fig.savefig(out / "critic_sanity.png", dpi=300, bbox_inches="tight")   # slide
    fig.savefig(out / "critic_sanity.pdf", bbox_inches="tight")            # report
    plt.show()
